# UniProt Fetch Examples

This notebook demonstrates protein entry retrieval from UniProt using **`run_uniprot_fetch`**.

- **Direct lookup** -- Fetch an entry by UniProt accession
- **Name search** -- Search by gene name and organism with ranked selection
- **Cross-references** -- Extract PDB structure links from UniProt entries

## Imports

In [ ]:
from bio_programming_tools.tools.database_retrieval import (
    run_uniprot_fetch,
    UniProtFetchInput,
    UniProtFetchConfig,
)

## API Reference

**`UniProtFetchInput`**

| Field | Type | Default | Description |
|---|---|---|---|
| `uniprot_id` | `Optional[str]` | `None` | UniProt accession for direct lookup |
| `target_name` | `Optional[str]` | `None` | Gene or protein name for search |
| `organism` | `Optional[str]` | `None` | Organism for search disambiguation |
| `prefer_pdb_crossref` | `bool` | `False` | Prefer entries with PDB cross-references |
| `max_candidates` | `int` | `5` | Max candidates for search ranking |

**`UniProtFetchConfig`**

| Field | Type | Default | Description |
|---|---|---|---|
| `request_timeout_seconds` | `int` | `15` | HTTP timeout |
| `http_retries` | `int` | `2` | Retries for HTTP requests |
| `backoff_seconds` | `float` | `1.0` | Base exponential backoff |

**`UniProtFetchOutput`**

| Field | Type | Description |
|---|---|---|
| `accession` | `str` | Primary UniProt accession |
| `sequence` | `Optional[str]` | Protein sequence |
| `length` | `Optional[int]` | Sequence length |
| `entry_type` | `Optional[str]` | UniProt entry type |
| `gene_names` | `List[str]` | Gene name symbols |
| `pdb_crossrefs` | `List[str]` | PDB cross-reference IDs |
| `source_url` | `str` | UniProt entry URL |
| `raw_entry` | `Dict[str, Any]` | Full UniProt JSON entry |

## 1. Fetch by UniProt Accession

Retrieve the TP53 protein entry directly by its UniProt accession.

In [ ]:
inputs = UniProtFetchInput(uniprot_id="P04637")

output = run_uniprot_fetch(inputs, UniProtFetchConfig())

print(f"Accession: {output.accession}")
print(f"Entry type: {output.entry_type}")
print(f"Gene names: {output.gene_names}")
print(f"Sequence length: {output.length}")
print(f"Preview: {output.sequence[:60]}...")
print(f"PDB cross-refs: {output.pdb_crossrefs[:5]} ({len(output.pdb_crossrefs)} total)")
print(f"Source: {output.source_url}")

## 2. Search by Gene Name and Organism

When you don't have a UniProt accession, search by gene name and organism. The tool ranks candidates by gene name match, reviewed status, and optionally PDB availability.

In [ ]:
inputs = UniProtFetchInput(
    target_name="lacI",
    organism="Escherichia coli",
)

output = run_uniprot_fetch(inputs, UniProtFetchConfig())

print(f"Accession: {output.accession}")
print(f"Gene names: {output.gene_names}")
print(f"Sequence length: {output.length}")
print(f"Preview: {output.sequence[:60]}...")

## 3. Search with PDB Cross-Reference Preference

Set `prefer_pdb_crossref=True` to prioritize entries that have linked PDB structures. Useful when you need both protein sequence and structure.

In [ ]:
inputs = UniProtFetchInput(
    target_name="insulin",
    organism="Homo sapiens",
    prefer_pdb_crossref=True,
    max_candidates=10,
)

output = run_uniprot_fetch(inputs, UniProtFetchConfig())

print(f"Accession: {output.accession}")
print(f"Gene names: {output.gene_names}")
print(f"Sequence length: {output.length}")
print(f"PDB cross-refs: {output.pdb_crossrefs[:5]}..." if len(output.pdb_crossrefs) > 5 else f"PDB cross-refs: {output.pdb_crossrefs}")
print(f"Total PDB links: {len(output.pdb_crossrefs)}")